In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import os

# --- PENGATURAN KONEKSI POSTGRESQL ---
DB_USER = 'postgres'
DB_PASS = 'postgres'  
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'olist_db'

connection_string = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(connection_string)

cleaned_path = '../data/cleaned/'

cleaned_files = {
    'customers': 'cleaned_customers.csv',
    'geolocation': 'cleaned_geolocation.csv',
    'order_items': 'cleaned_order_items.csv',
    'order_payments': 'cleaned_order_payments.csv',
    'order_reviews': 'cleaned_order_reviews.csv',
    'orders': 'cleaned_orders.csv',
    'products': 'cleaned_products.csv',
    'sellers': 'cleaned_sellers.csv',
    'category_translation': 'cleaned_category_translation.csv'
}

print("Memulai upload data ke PostgreSQL...")

for table_name, file_name in cleaned_files.items():
    file_full_path = os.path.join(cleaned_path, file_name)
    df = pd.read_csv(file_full_path)
    
    # Upload ke PostgreSQL
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f"✅ Tabel '{table_name}' berhasil dimuat ({len(df)} baris)")

print("\n  Semua data Olist berhasil tersimpan di Database PostgreSQL 'olist_db'!")

Memulai upload data ke PostgreSQL...
✅ Tabel 'customers' berhasil dimuat (99441 baris)
✅ Tabel 'geolocation' berhasil dimuat (19015 baris)
✅ Tabel 'order_items' berhasil dimuat (112650 baris)
✅ Tabel 'order_payments' berhasil dimuat (103886 baris)
✅ Tabel 'order_reviews' berhasil dimuat (99224 baris)
✅ Tabel 'orders' berhasil dimuat (99441 baris)
✅ Tabel 'products' berhasil dimuat (32951 baris)
✅ Tabel 'sellers' berhasil dimuat (3095 baris)
✅ Tabel 'category_translation' berhasil dimuat (71 baris)

🎉 SELAMAT! Semua data Olist berhasil tersimpan di Database PostgreSQL 'olist_db'!


In [4]:
# Query SQL untuk melihat total revenue dan tren bulanan
query_monthly = """
SELECT 
    TO_CHAR(o.order_purchase_timestamp::timestamp, 'YYYY-MM') AS order_month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(i.price)::numeric, 2) AS monthly_revenue
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
GROUP BY order_month
ORDER BY order_month;
"""

# Menjalankan query SQL ke database PostgreSQL menggunakan pandas
df_monthly = pd.read_sql_query(query_monthly, engine)

# Menampilkan hasil tabelnya
df_monthly.head(10)

,order_month,total_orders,monthly_revenue
0,2016-09,1,134.97
1,2016-10,265,40325.11
2,2016-12,1,10.90
3,2017-01,750,111798.36
4,2017-02,1653,234223.40
5,2017-03,2546,359198.85
6,2017-04,2303,340669.68
7,2017-05,3546,489338.25
8,2017-06,3135,421923.37
9,2017-07,3872,481604.52


#Product Category Analysis

In [5]:
query_top_categories = """
SELECT 
    COALESCE(p.product_category_name_english, 'Unknown') AS category,
    COUNT(DISTINCT i.order_id) AS total_orders_sold,
    COUNT(i.product_id) AS total_items_sold,
    ROUND(SUM(i.price)::numeric, 2) AS total_revenue
FROM order_items i
JOIN products p ON i.product_id = p.product_id
JOIN orders o ON i.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10;
"""

df_top_categories = pd.read_sql_query(query_top_categories, engine)
df_top_categories

,category,total_orders_sold,total_items_sold,total_revenue
0,health_beauty,8647,9465,1233131.72
1,watches_gifts,5495,5859,1166176.98
2,bed_bath_table,9272,10953,1023434.76
3,sports_leisure,7530,8431,954852.55
4,computers_accessories,6530,7644,888724.61
5,furniture_decor,6307,8160,711927.69
6,housewares,5743,6795,615628.69
7,cool_stuff,3559,3718,610204.10
8,auto,3810,4140,578966.65
9,toys,3804,4030,471286.48


#Logistics & Delivery Performance

In [6]:
query_logistics = """
SELECT 
    COUNT(order_id) AS total_delivered_orders,
    ROUND(AVG(EXTRACT(DAY FROM (order_delivered_customer_date::timestamp - order_purchase_timestamp::timestamp)))::numeric, 1) AS avg_delivery_days,
    COUNT(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 END) AS late_orders,
    ROUND((COUNT(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 END)::numeric / COUNT(order_id) * 100), 2) AS late_rate_pct
FROM orders
WHERE order_status = 'delivered';
"""

df_logistics = pd.read_sql_query(query_logistics, engine)
df_logistics

,total_delivered_orders,avg_delivery_days,late_orders,late_rate_pct
0,96478,12.1,7826,8.11


#Impact of Late Delivery on Review Scores

In [7]:
query_review_impact = """
SELECT 
    CASE 
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 'Terlambat (Late)'
        ELSE 'Tepat Waktu (On-Time)'
    END AS delivery_status,
    COUNT(r.review_id) AS total_reviews,
    ROUND(AVG(r.review_score)::numeric, 2) AS avg_review_score
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY delivery_status;
"""

df_review_impact = pd.read_sql_query(query_review_impact, engine)
df_review_impact

,delivery_status,total_reviews,avg_review_score
0,Tepat Waktu (On-Time),88661,4.29
1,Terlambat (Late),7700,2.57


#Payment Method Preferences

In [8]:
query_payment = """
SELECT 
    payment_type,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(payment_value)::numeric, 2) AS total_payment_value,
    ROUND(AVG(payment_installments)::numeric, 1) AS avg_installments
FROM order_payments
GROUP BY payment_type
ORDER BY total_payment_value DESC;
"""

df_payment = pd.read_sql_query(query_payment, engine)
df_payment

,payment_type,total_orders,total_payment_value,avg_installments
0,credit_card,76505,12542084.19,3.5
1,boleto,19784,2869361.27,1.0
2,voucher,3866,379436.87,1.0
3,debit_card,1528,217989.79,1.0
4,not_defined,3,0.00,1.0
